# 00 — DualPath Pipeline: Vehicle Detection + Geometric Lane Prediction

**Architecture: DualPathNet**
- Shared ResNet-50 + FPN backbone
- RT-DETR-inspired transformer detection decoder (vehicle-only)
- MapTR-inspired query-based lane decoder (structured polylines, NOT masks)
- Optional weak cross-branch attention

**Dataset: BDD100K**
- Detection: YOLO-format labels, vehicle classes only
- Lanes: structured targets from original BDD100K poly2d annotations

**Key design choices:**
- Weakly-coupled dual-path reduces negative transfer between tasks
- Lane prediction is geometric (ordered point sequences), not raster masks
- Architecture is future-ready for temporal lane memory (StreamMapNet-style)

## 1 — Environment Setup

In [ ]:
!pip install -q torch torchvision torchmetrics pyyaml scipy opencv-python matplotlib tqdm

In [ ]:
import torch
import os, sys

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu} ({mem:.1f} GB)")
else:
    print("WARNING: No GPU — training will be slow")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ── Fixed Drive paths (preserved from old pipeline) ──
ECOCAR_ROOT = "/content/drive/MyDrive/EcoCAR"
DATASET_DRIVE = os.path.join(ECOCAR_ROOT, "datasets", "bdd100k_yolo")
WEIGHTS_DIR = os.path.join(ECOCAR_ROOT, "weights")
TRAINING_RUNS = os.path.join(ECOCAR_ROOT, "training_runs")

# ── Project code ──
PROJECT_DIR = os.path.join(ECOCAR_ROOT, "DETR_GeoLane_pipeline")
if not os.path.isdir(PROJECT_DIR):
    # Fallback: clone from repo
    !git clone https://github.com/ChenSiyun1234/EcoCAR-Perception-Pipeline-YOLO26-BDD100K.git /content/repo 2>/dev/null || true
    PROJECT_DIR = "/content/repo/DETR_GeoLane_pipeline"

sys.path.insert(0, PROJECT_DIR)
print(f"Project: {PROJECT_DIR}")
print(f"Dataset: {DATASET_DRIVE}")

## 2 — Dataset Inspection

Reuses the existing BDD100K YOLO-format dataset from the old pipeline.
Detection labels are `.txt` files; lane targets are parsed from raw BDD100K JSON.

In [ ]:
import tarfile

LOCAL_DS = "/content/bdd100k_yolo"
os.makedirs(LOCAL_DS, exist_ok=True)

# Extract from Drive tar if needed (faster than FUSE)
NB02_TAR = os.path.join(ECOCAR_ROOT, "datasets", "bdd100k_yolo_nb02.tar")
if not os.path.isdir(os.path.join(LOCAL_DS, "images", "val")):
    if os.path.isfile(NB02_TAR):
        print(f"Extracting {NB02_TAR} ...")
        with tarfile.open(NB02_TAR, "r") as tar:
            tar.extractall(LOCAL_DS, filter='data')
        print("Done.")
    elif os.path.isdir(os.path.join(DATASET_DRIVE, "images")):
        # Symlink from Drive
        for sub in ["images", "labels"]:
            src = os.path.join(DATASET_DRIVE, sub)
            dst = os.path.join(LOCAL_DS, sub)
            if not os.path.exists(dst) and os.path.isdir(src):
                os.symlink(src, dst)
        print("Linked dataset from Drive.")
    else:
        print(f"ERROR: No dataset found at {NB02_TAR} or {DATASET_DRIVE}")

# Count
for split in ["train", "val"]:
    img_dir = os.path.join(LOCAL_DS, "images", split)
    lbl_dir = os.path.join(LOCAL_DS, "labels", split)
    n_img = len(os.listdir(img_dir)) if os.path.isdir(img_dir) else 0
    n_lbl = len(os.listdir(lbl_dir)) if os.path.isdir(lbl_dir) else 0
    print(f"  {split}: {n_img} images, {n_lbl} labels")

In [ ]:
# ── Detection class distribution ──
from collections import Counter

from src.config import BDD_TO_VEHICLE, VEHICLE_CLASSES

lbl_dir = os.path.join(LOCAL_DS, "labels", "train")
class_counts = Counter()
total_files = 0

if os.path.isdir(lbl_dir):
    for f in os.listdir(lbl_dir):
        if not f.endswith(".txt"):
            continue
        total_files += 1
        with open(os.path.join(lbl_dir, f)) as fh:
            for line in fh:
                cls_id = int(line.strip().split()[0])
                if cls_id in BDD_TO_VEHICLE:
                    class_counts[BDD_TO_VEHICLE[cls_id]] += 1

print(f"Vehicle detection class distribution ({total_files} files):")
for i, name in enumerate(VEHICLE_CLASSES):
    print(f"  {name:<12}: {class_counts.get(i, 0):>8,}")
print(f"  Total vehicles: {sum(class_counts.values()):>8,}")

## 3 — Lane Annotation Parsing

Parse raw BDD100K poly2d annotations into structured lane targets.
Each lane becomes a fixed-length ordered point sequence (72 points).

In [ ]:
from src.config import find_lane_labels

train_lane_json = find_lane_labels("train")
val_lane_json = find_lane_labels("val")

print(f"Train lane labels: {train_lane_json or 'NOT FOUND'}")
print(f"Val lane labels:   {val_lane_json or 'NOT FOUND'}")

if train_lane_json is None:
    print()
    print("="*60)
    print("Lane labels not found on Drive.")
    print("To enable lane training, download the BDD100K consolidated")
    print("label files and place them in one of these locations:")
    print("  - /content/drive/MyDrive/EcoCAR/datasets/bdd100k/labels/")
    print("  - /content/drive/MyDrive/EcoCAR/datasets/")
    print()
    print("Expected filenames:")
    print("  - bdd100k_labels_images_train.json")
    print("  - bdd100k_labels_images_val.json")
    print()
    print("Download from: https://bdd-data.berkeley.edu/")
    print("The pipeline will still train detection without lane labels.")
    print("="*60)

In [ ]:
# Preview lane annotations if available
if train_lane_json:
    import json
    from src.lane_targets import LaneLabelCache, frame_to_lane_targets

    cache = LaneLabelCache(train_lane_json, max_lanes=10, num_points=72)

    # Show stats
    n_with_lanes = len(cache)
    print(f"Frames with lane annotations: {n_with_lanes}")

    # Preview one frame
    import matplotlib.pyplot as plt
    import numpy as np

    for name in list(cache._cache.keys())[:1]:
        targets = cache.get(name)
        n_lanes = int(targets["existence"].sum())
        print(f"\nSample: {name} — {n_lanes} lanes")

        fig, ax = plt.subplots(figsize=(10, 3))
        for i in range(n_lanes):
            pts = targets["points"][i]
            ax.plot(pts[:, 0] * 1280, pts[:, 1] * 720, '-', linewidth=2, label=f"Lane {i}")
        ax.set_xlim(0, 1280)
        ax.set_ylim(720, 0)
        ax.set_title(f"Lane polylines: {name}")
        ax.legend()
        plt.show()

## 4 — Configuration

In [ ]:
from src.config import Config

cfg = Config(
    run_name="dualpath_v1",
    dataset_root=LOCAL_DS,
    img_size=640,
    batch_size=8,
    backbone="resnet50",
    pretrained=True,
    det_num_queries=100,
    det_dec_layers=3,
    lane_num_queries=10,
    lane_dec_layers=3,
    lane_points=72,
    cross_attn=True,
    epochs=50,
    lr=1e-4,
    patience=15,
)

cfg.save(os.path.join(cfg.save_dir, "config.yaml"))
print(f"Config saved to {cfg.save_dir}")
print(f"  Backbone: {cfg.backbone}")
print(f"  Det queries: {cfg.det_num_queries}")
print(f"  Lane queries: {cfg.lane_num_queries} x {cfg.lane_points} points")
print(f"  Cross-attn: {cfg.cross_attn}")

## 5 — Build Model

In [ ]:
from src.model import build_model

model = build_model(cfg)
model.print_summary()

In [ ]:
# Sanity check forward pass
dummy = torch.randn(2, 3, cfg.img_size, cfg.img_size)
if torch.cuda.is_available():
    dummy = dummy.cuda()
    model = model.cuda()

with torch.no_grad():
    out = model(dummy)

for k, v in out.items():
    if isinstance(v, torch.Tensor):
        print(f"  {k}: {v.shape}")

model = model.cpu()
torch.cuda.empty_cache()

## 6 — Dataset & DataLoader

In [ ]:
from src.dataset import build_dataloaders

train_loader, val_loader = build_dataloaders(
    cfg,
    train_lane_json=train_lane_json,
    val_lane_json=val_lane_json,
)

print(f"Train: {len(train_loader.dataset)} samples, {len(train_loader)} batches")
print(f"Val:   {len(val_loader.dataset)} samples, {len(val_loader)} batches")

In [ ]:
# Visualize a batch
import matplotlib.pyplot as plt
import numpy as np

batch = next(iter(train_loader))
print(f"Images: {batch['images'].shape}")
print(f"Det targets: {batch['det_targets'].shape}")
print(f"Lane existence: {batch['lane_existence'].shape}")
print(f"Lane points: {batch['lane_points'].shape}")
print(f"Has lanes: {batch['has_lanes']}")

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for i in range(min(4, batch['images'].shape[0])):
    img = batch['images'][i].permute(1, 2, 0).numpy()
    img = np.clip(img, 0, 1)
    axes[i].imshow(img)
    axes[i].set_title(f"Image {i}")
    axes[i].axis('off')
plt.tight_layout()
plt.show()

## 7 — Training

Trains the DualPathNet with:
- AdamW optimizer (backbone LR x0.1)
- Cosine LR schedule with warmup
- Hungarian matching for both detection and lane tasks
- Metric-driven checkpointing (best det, best lane, best joint)

In [ ]:
from src.trainer import Trainer

trainer = Trainer(cfg, model, train_loader, val_loader)
print(f"Training for {cfg.epochs} epochs")
print(f"Save dir: {cfg.save_dir}")

In [ ]:
import traceback

try:
    history = trainer.train()
except Exception as e:
    print(f"\nTraining stopped: {e}")
    traceback.print_exc(limit=5)
    history = trainer.history

print(f"\nLogged {len(history)} epochs.")

## 8 — Training Curves

In [ ]:
import matplotlib.pyplot as plt

if history and len(history) > 0:
    epochs = range(1, len(history) + 1)
    train_loss = [h.get("train_loss", 0) for h in history]
    val_loss = [h.get("val_loss", 0) for h in history]
    det_map = [h.get("det_map50", 0) * 100 for h in history]
    lane_f1 = [h.get("lane_f1", 0) * 100 for h in history]

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    axes[0].plot(epochs, train_loss, label="Train")
    axes[0].plot(epochs, val_loss, label="Val")
    axes[0].set_title("Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()

    axes[1].plot(epochs, det_map)
    axes[1].set_title("Detection mAP@50 (%)")
    axes[1].set_xlabel("Epoch")

    axes[2].plot(epochs, lane_f1)
    axes[2].set_title("Lane F1 (%)")
    axes[2].set_xlabel("Epoch")

    plt.tight_layout()
    plt.savefig(os.path.join(cfg.save_dir, "training_curves.png"), dpi=150)
    plt.show()
else:
    print("No history to plot.")

## 9 — Qualitative Inference

In [ ]:
import cv2
import random
from src.visualize import draw_all

model.eval()
if torch.cuda.is_available():
    model = model.cuda()

val_dir = os.path.join(LOCAL_DS, "images", "val")
all_images = sorted(os.listdir(val_dir))
random.seed(42)
sample_images = random.sample(all_images, min(6, len(all_images)))

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
for i, fname in enumerate(sample_images):
    img_bgr = cv2.imread(os.path.join(val_dir, fname))
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img_rgb, (cfg.img_size, cfg.img_size))

    tensor = torch.from_numpy(img_resized.astype(np.float32) / 255.0).permute(2, 0, 1)
    tensor = tensor.unsqueeze(0).to(next(model.parameters()).device)

    with torch.no_grad(), torch.amp.autocast("cuda", enabled=cfg.amp):
        outputs = model(tensor)

    vis = draw_all(img_resized, outputs, conf_thresh=0.3, lane_thresh=0.5)

    ax = axes[i // 3, i % 3]
    ax.imshow(vis)
    ax.set_title(fname)
    ax.axis('off')

plt.suptitle("DualPathNet Inference", fontsize=14)
plt.tight_layout()
plt.show()

## 10 — Export Best Weights

In [ ]:
import shutil

best_src = os.path.join(cfg.save_dir, "weights", "best_joint.pt")
best_dst = os.path.join(WEIGHTS_DIR, f"{cfg.run_name}_best.pt")

os.makedirs(WEIGHTS_DIR, exist_ok=True)

if os.path.isfile(best_src):
    shutil.copy2(best_src, best_dst)
    print(f"Best weights copied to: {best_dst}")
else:
    last_src = os.path.join(cfg.save_dir, "weights", "last.pt")
    if os.path.isfile(last_src):
        last_dst = os.path.join(WEIGHTS_DIR, f"{cfg.run_name}_last.pt")
        shutil.copy2(last_src, last_dst)
        print(f"Last weights copied to: {last_dst}")

print("\n" + "="*55)
print("  TRAINING COMPLETE")
print("="*55)
print(f"  Run:        {cfg.run_name}")
print(f"  Epochs:     {len(history)}")
print(f"  Best det:   {trainer.best_scores['det']*100:.2f}%")
print(f"  Best lane:  {trainer.best_scores['lane']*100:.2f}%")
print(f"  Best joint: {trainer.best_scores['joint']:.4f}")
print(f"  Weights:    {WEIGHTS_DIR}")
print("="*55)